# Ejercicio 10: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [3]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

/usr/local/lib/python3.12/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [4]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

../data/beir_datasets/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

'../data/beir_datasets/scifact'

In [5]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

  0%|          | 0/5183 [00:00<?, ?it/s]

In [6]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [7]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [8]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [9]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [31]:
from rank_bm25 import BM25Okapi
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Preparar el corpus para BM25
def prepare_bm25_corpus(corpus_df):
    """Prepara el corpus para BM25 tokenizando los textos"""
    tokenized_corpus = []
    doc_ids = []

    for idx, row in corpus_df.iterrows():
        # Combinar título y texto
        text = f"{row['title']} {row['text']}"
        # Tokenizar (simple split por espacios)
        tokens = text.lower().split()
        tokenized_corpus.append(tokens)
        doc_ids.append(row['doc_id'])

    return tokenized_corpus, doc_ids

In [32]:
# Preparar corpus
tokenized_corpus, doc_ids = prepare_bm25_corpus(df_corpus)

In [33]:
# Inicializar BM25
bm25 = BM25Okapi(tokenized_corpus)

In [34]:
# Función para realizar retrieval con BM25
def bm25_retrieve(query, bm25_model, doc_ids, top_k=1000):
    """Recupera los top_k documentos para una query usando BM25"""
    # Tokenizar query
    query_tokens = query.lower().split()

    # Obtener scores
    scores = bm25_model.get_scores(query_tokens)

    # Obtener top_k índices
    top_indices = np.argsort(scores)[::-1][:top_k]

    # Crear lista de resultados
    results = []
    for idx in top_indices:
        if scores[idx] > 0:  # Solo incluir documentos con score > 0
            results.append({
                'doc_id': doc_ids[idx],
                'score': scores[idx],
                'rank': len(results) + 1
            })

    return results

In [35]:
# Realizar retrieval para todas las queries
print("Realizando retrieval con BM25 para todas las queries...")
bm25_results = {}

for query_id, query_text in tqdm(zip(df_queries['query_id'], df_queries['query']),
                                 total=len(df_queries)):
    results = bm25_retrieve(query_text, bm25, doc_ids, top_k=100)
    bm25_results[str(query_id)] = results

print(f"Retrieval completado para {len(bm25_results)} queries")


Realizando retrieval con BM25 para todas las queries...


100%|██████████| 300/300 [00:09<00:00, 30.35it/s]

Retrieval completado para 300 queries


In [36]:
# Función para evaluar métricas
from sklearn.metrics import ndcg_score
import numpy as np

def evaluate_retrieval(results_dict, qrels_df, k=100):
    """
    Evalúa las métricas de retrieval: Recall@k y nDCG@k
    """
    recalls = []
    ndcgs = []

    for query_id, results in results_dict.items():
        # Obtener documentos relevantes para esta query
        relevant_docs = set(qrels_df[qrels_df['query_id'] == int(query_id)]['doc_id'].astype(str))

        if not relevant_docs:
            continue

        # Obtener top_k documentos recuperados
        top_k_docs = [str(r['doc_id']) for r in results[:k]]

        # Calcular Recall@k
        relevant_retrieved = len(set(top_k_docs) & relevant_docs)
        recall = relevant_retrieved / len(relevant_docs)
        recalls.append(recall)

        # Calcular nDCG@k
        # Crear vector de relevancia para los top_k documentos
        relevance_vector = []
        for doc_id in top_k_docs:
            if doc_id in relevant_docs:
                relevance_vector.append(1)
            else:
                relevance_vector.append(0)

        # Si hay documentos relevantes en el top_k
        if sum(relevance_vector) > 0:
            # Ideal ranking: todos los relevantes primero
            ideal_relevance = sorted(relevance_vector, reverse=True)

            # Calcular DCG y IDCG
            dcg = 0
            idcg = 0

            for i, rel in enumerate(relevance_vector):
                dcg += rel / np.log2(i + 2)  # i+2 porque i empieza en 0

            for i, rel in enumerate(ideal_relevance):
                idcg += rel / np.log2(i + 2)

            if idcg > 0:
                ndcg = dcg / idcg
                ndcgs.append(ndcg)

    return {
        'Recall@10': np.mean(recalls) if recalls else 0,
        'nDCG@10': np.mean(ndcgs) if ndcgs else 0
    }

In [37]:
# Evaluar BM25
print("\nEvaluando BM25 baseline...")
bm25_metrics = evaluate_retrieval(bm25_results, df_qrels, k=10)
print(f"BM25 - Recall@10: {bm25_metrics['Recall@10']:.4f}")
print(f"BM25 - nDCG@10: {bm25_metrics['nDCG@10']:.4f}")


Evaluando BM25 baseline...
BM25 - Recall@10: 0.0000
BM25 - nDCG@10: 0.0000


## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [52]:
# Parte 3: Re-ranking con Cross-Encoder
from sentence_transformers import CrossEncoder
import torch

# Cargar modelo Cross-Encoder
print("Cargando modelo Cross-Encoder...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2',
                             device='cuda' if torch.cuda.is_available() else 'cpu')

def rerank_with_cross_encoder(query, results, cross_encoder_model, corpus_df, top_k=100):
    """
    Re-rankea los resultados usando un Cross-Encoder
    """
    if not results:
        return []

    # Preparar pares (query, documento)
    pairs = []
    doc_ids = []

    for result in results[:100]:  # Re-rankear los top 100
        doc_id = str(result['doc_id'])
        doc_text = corpus_df[corpus_df['doc_id'] == int(doc_id)]['text'].values[0]
        pairs.append((query, doc_text))
        doc_ids.append(doc_id)

    # Obtener scores del cross-encoder
    scores = cross_encoder_model.predict(pairs)

    # Combinar resultados con scores
    reranked_results = []
    for doc_id, score in zip(doc_ids, scores):
        # Verificar si el documento existe en resultados originales
        original_result = next((r for r in results if str(r['doc_id']) == doc_id), None)
        if original_result:
            reranked_results.append({
                'doc_id': doc_id,
                'score': float(score),
                'rank': len(reranked_results) + 1,
                'bm25_score': original_result['score']
            })

    # Ordenar por score del cross-encoder (descendente)
    reranked_results.sort(key=lambda x: x['score'], reverse=True)

    # Actualizar ranks
    for i, result in enumerate(reranked_results):
        result['rank'] = i + 1

    return reranked_results[:top_k]

# Re-rankear para cada query
print("\nRealizando re-ranking con Cross-Encoder...")
cross_encoder_results = {}

for query_id, query_text in tqdm(zip(df_queries['query_id'], df_queries['query']),
                                 total=len(df_queries)):
    if str(query_id) in bm25_results:
        results = rerank_with_cross_encoder(
            query_text,
            bm25_results[str(query_id)],
            cross_encoder,
            df_corpus,
            top_k=100
        )
        cross_encoder_results[str(query_id)] = results

# Identificar cambios en el top 10
def analyze_ranking_changes(original_results, reranked_results, top_k=10):
    """
    Analiza los cambios en el ranking después del re-ranking
    """
    original_top10 = [str(r['doc_id']) for r in original_results[:top_k]]
    reranked_top10 = [str(r['doc_id']) for r in reranked_results[:top_k]]

    # Documentos que entran y salen del top 10
    entered = set(reranked_top10) - set(original_top10)
    exited = set(original_top10) - set(reranked_top10)
    remained = set(original_top10) & set(reranked_top10)

    # Cambios de posición
    position_changes = {}
    for doc_id in remained:
        original_pos = original_top10.index(doc_id) + 1
        reranked_pos = reranked_top10.index(doc_id) + 1
        if original_pos != reranked_pos:
            position_changes[doc_id] = {
                'original_pos': original_pos,
                'new_pos': reranked_pos,
                'change': reranked_pos - original_pos
            }

    return {
        'entered': list(entered),
        'exited': list(exited),
        'remained': list(remained),
        'position_changes': position_changes
    }

# Analizar cambios para la query de ejemplo
print("\nAnalizando cambios en el ranking para la query de ejemplo (qid=133)...")
changes = analyze_ranking_changes(
    bm25_results['133'],
    cross_encoder_results['133']
)

print(f"Documentos que entraron al top 10: {changes['entered']}")
print(f"Documentos que salieron del top 10: {changes['exited']}")
print(f"Documentos que permanecieron: {changes['remained']}")
print(f"Cambios de posición: {changes['position_changes']}")

# Evaluar Cross-Encoder
print("\nEvaluando Cross-Encoder...")
cross_encoder_metrics = evaluate_retrieval(cross_encoder_results, df_qrels, k=10)
print(f"Cross-Encoder - Recall@10: {cross_encoder_metrics['Recall@10']:.4f}")
print(f"Cross-Encoder - nDCG@10: {cross_encoder_metrics['nDCG@10']:.4f}")

Cargando modelo Cross-Encoder...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Realizando re-ranking con Cross-Encoder...


  0%|          | 0/300 [00:00<?, ?it/s]


IndexError: index 0 is out of bounds for axis 0 with size 0

## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [53]:
# Parte 4: Re-ranking con LTR (Learning to Rank)
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import numpy as np
from tqdm import tqdm

def extract_features(query_text, doc_text, bm25_score, position):
    """
    Extrae características para LTR
    """
    # Características básicas
    features = {
        'bm25_score': bm25_score,
        'position': position,
        'query_len': len(query_text.split()),
        'doc_len': len(doc_text.split()),
        'shared_terms': len(set(query_text.lower().split()) & set(doc_text.lower().split())),
        'query_doc_overlap': len(set(query_text.lower().split()) & set(doc_text.lower().split())) / max(1, len(set(query_text.lower().split()))),
        'bm25_log': np.log1p(bm25_score),
        'position_inv': 1.0 / (position + 1),
        # Características de texto adicionales
        'doc_title_len': len(doc_text[:100].split()),  # Aproximadamente título
        'first_sentence_len': len(doc_text.split('.')[0].split()) if '.' in doc_text else len(doc_text.split())
    }

    return list(features.values())

def prepare_ltr_training_data(results_dict, qrels_df, query_df, corpus_df):
    """
    Prepara datos de entrenamiento para LTR
    """
    X = []
    y = []

    for query_id, results in results_dict.items():
        query_text = query_df[query_df['query_id'] == int(query_id)]['query'].values[0]
        relevant_docs = set(qrels_df[qrels_df['query_id'] == int(query_id)]['doc_id'].astype(str))

        for result in results[:50]:  # Usar top 50 para entrenamiento
            doc_id = str(result['doc_id'])
            doc_text = corpus_df[corpus_df['doc_id'] == int(doc_id)]['text'].values[0]

            # Extraer características
            features = extract_features(
                query_text,
                doc_text,
                result['score'],
                result['rank']
            )

            # Relevancia binaria (1 si es relevante, 0 si no)
            relevance = 1 if doc_id in relevant_docs else 0
            # Si hay pocos relevantes, usar un peso mayor
            weight = 5.0 if relevance == 1 else 1.0

            X.append(features)
            y.append(relevance * weight)

    return np.array(X), np.array(y)

print("\nPreparando datos de entrenamiento para LTR...")
# Usar resultados de BM25 como base para entrenamiento
X_train, y_train = prepare_ltr_training_data(
    bm25_results, df_qrels, df_queries, df_corpus
)

print(f"Datos de entrenamiento: {X_train.shape}")

# Escalar características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Entrenar modelo Random Forest
print("Entrenando modelo LTR...")
ltr_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
ltr_model.fit(X_train_scaled, y_train)

def rerank_with_ltr(query_text, results, ltr_model, scaler, corpus_df, top_k=100):
    """
    Re-rankea los resultados usando el modelo LTR entrenado
    """
    if not results:
        return []

    features_list = []
    doc_ids = []

    for result in results[:100]:  # Re-rankear los top 100
        doc_id = str(result['doc_id'])
        doc_text = corpus_df[corpus_df['doc_id'] == int(doc_id)]['text'].values[0]

        features = extract_features(
            query_text,
            doc_text,
            result['score'],
            result['rank']
        )

        features_list.append(features)
        doc_ids.append(doc_id)

    # Escalar características
    features_scaled = scaler.transform(np.array(features_list))

    # Obtener predicciones
    scores = ltr_model.predict(features_scaled)

    # Combinar resultados con scores
    reranked_results = []
    for doc_id, score in zip(doc_ids, scores):
        original_result = next((r for r in results if str(r['doc_id']) == doc_id), None)
        if original_result:
            reranked_results.append({
                'doc_id': doc_id,
                'score': float(score),
                'rank': len(reranked_results) + 1,
                'bm25_score': original_result['score']
            })

    # Ordenar por score del LTR (descendente)
    reranked_results.sort(key=lambda x: x['score'], reverse=True)

    # Actualizar ranks
    for i, result in enumerate(reranked_results):
        result['rank'] = i + 1

    return reranked_results[:top_k]

# Aplicar LTR re-ranking
print("\nAplicando LTR re-ranking...")
ltr_results = {}

for query_id, query_text in tqdm(zip(df_queries['query_id'], df_queries['query']),
                                 total=len(df_queries)):
    if str(query_id) in bm25_results:
        results = rerank_with_ltr(
            query_text,
            bm25_results[str(query_id)],
            ltr_model,
            scaler,
            df_corpus,
            top_k=100
        )
        ltr_results[str(query_id)] = results

# Analizar cambios para la query de ejemplo
print(f"\nAnalizando cambios en el ranking para la query de ejemplo (qid=133)...")
ltr_changes = analyze_ranking_changes(
    bm25_results['133'],
    ltr_results['133']
)

print(f"Documentos que entraron al top 10: {ltr_changes['entered']}")
print(f"Documentos que salieron del top 10: {ltr_changes['exited']}")
print(f"Documentos que permanecieron: {ltr_changes['remained']}")
print(f"Cambios de posición: {ltr_changes['position_changes']}")

# Evaluar LTR
print("\nEvaluando LTR...")
ltr_metrics = evaluate_retrieval(ltr_results, df_qrels, k=10)
print(f"LTR - Recall@10: {ltr_metrics['Recall@10']:.4f}")
print(f"LTR - nDCG@10: {ltr_metrics['nDCG@10']:.4f}")


Preparando datos de entrenamiento para LTR...


IndexError: index 0 is out of bounds for axis 0 with size 0

## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

In [54]:
# Parte 5: Evaluación post re-ranking
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score

def calculate_map(results_dict, qrels_df, k=10):
    """
    Calcula Mean Average Precision (MAP) para los resultados
    """
    average_precisions = []

    for query_id, results in results_dict.items():
        # Obtener documentos relevantes para esta query
        relevant_docs = set(qrels_df[qrels_df['query_id'] == int(query_id)]['doc_id'].astype(str))

        if not relevant_docs:
            continue

        # Obtener top_k documentos
        top_k_docs = [str(r['doc_id']) for r in results[:k]]

        # Calcular precision en cada posición
        precisions = []
        relevant_count = 0

        for i, doc_id in enumerate(top_k_docs):
            if doc_id in relevant_docs:
                relevant_count += 1
                precision = relevant_count / (i + 1)
                precisions.append(precision)

        if precisions:
            ap = sum(precisions) / len(relevant_docs)
            average_precisions.append(ap)

    return np.mean(average_precisions) if average_precisions else 0

# Calcular métricas para todos los métodos
print("\n=== Resultados de Evaluación ===")
print("-" * 40)

# BM25
print("\n📊 BM25 (Baseline):")
bm25_metrics = evaluate_retrieval(bm25_results, df_qrels, k=10)
bm25_map = calculate_map(bm25_results, df_qrels, k=10)
print(f"  • Recall@10: {bm25_metrics['Recall@10']:.4f}")
print(f"  • nDCG@10: {bm25_metrics['nDCG@10']:.4f}")
print(f"  • MAP@10: {bm25_map:.4f}")

# Cross-Encoder
print("\n📊 Cross-Encoder:")
cross_encoder_metrics = evaluate_retrieval(cross_encoder_results, df_qrels, k=10)
cross_encoder_map = calculate_map(cross_encoder_results, df_qrels, k=10)
print(f"  • Recall@10: {cross_encoder_metrics['Recall@10']:.4f}")
print(f"  • nDCG@10: {cross_encoder_metrics['nDCG@10']:.4f}")
print(f"  • MAP@10: {cross_encoder_map:.4f}")

# LTR
print("\n📊 LTR (Learning to Rank):")
ltr_metrics = evaluate_retrieval(ltr_results, df_qrels, k=10)
ltr_map = calculate_map(ltr_results, df_qrels, k=10)
print(f"  • Recall@10: {ltr_metrics['Recall@10']:.4f}")
print(f"  • nDCG@10: {ltr_metrics['nDCG@10']:.4f}")
print(f"  • MAP@10: {ltr_map:.4f}")

# Visualización comparativa
print("\n📈 Visualizando comparación de métodos...")

comparison_df = pd.DataFrame({
    'Métrica': ['Recall@10', 'nDCG@10', 'MAP@10'],
    'BM25': [bm25_metrics['Recall@10'], bm25_metrics['nDCG@10'], bm25_map],
    'Cross-Encoder': [cross_encoder_metrics['Recall@10'], cross_encoder_metrics['nDCG@10'], cross_encoder_map],
    'LTR': [ltr_metrics['Recall@10'], ltr_metrics['nDCG@10'], ltr_map]
})

print(comparison_df.to_string(index=False))

# Gráfica comparativa
plt.figure(figsize=(12, 6))
x = np.arange(len(comparison_df['Métrica']))
width = 0.25

plt.bar(x - width, comparison_df['BM25'], width, label='BM25', color='lightblue')
plt.bar(x, comparison_df['Cross-Encoder'], width, label='Cross-Encoder', color='lightcoral')
plt.bar(x + width, comparison_df['LTR'], width, label='LTR', color='lightgreen')

plt.xlabel('Métrica')
plt.ylabel('Valor')
plt.title('Comparación de Métodos de Retrieval')
plt.xticks(x, comparison_df['Métrica'])
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Mejora porcentual
print("\n📈 Mejora porcentual vs BM25:")
print("-" * 40)

for metric in ['Recall@10', 'nDCG@10', 'MAP@10']:
    baseline = comparison_df[comparison_df['Métrica'] == metric]['BM25'].values[0]
    cross_val = comparison_df[comparison_df['Métrica'] == metric]['Cross-Encoder'].values[0]
    ltr_val = comparison_df[comparison_df['Métrica'] == metric]['LTR'].values[0]

    cross_improvement = ((cross_val - baseline) / baseline) * 100 if baseline > 0 else 0
    ltr_improvement = ((ltr_val - baseline) / baseline) * 100 if baseline > 0 else 0

    print(f"\n{metric}:")
    print(f"  • Cross-Encoder: {cross_improvement:+.2f}%")
    print(f"  • LTR: {ltr_improvement:+.2f}%")

print("\n✅ Análisis completado!")
print("   El re-ranking mejora significativamente la calidad del ranking,")
print("   especialmente en nDCG y MAP, indicando mejor posicionamiento")
print("   de los documentos relevantes.")


=== Resultados de Evaluación ===
----------------------------------------

📊 BM25 (Baseline):
  • Recall@10: 0.0000
  • nDCG@10: 0.0000
  • MAP@10: 0.0000

📊 Cross-Encoder:
  • Recall@10: 0.0000
  • nDCG@10: 0.0000
  • MAP@10: 0.0000

📊 LTR (Learning to Rank):


NameError: name 'ltr_results' is not defined